In [20]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel

In [21]:
train = pd.read_pickle("train_final.pkl")
val   = pd.read_pickle("val_final.pkl")

print(train.shape, val.shape)
print(train.columns)

(1971, 15) (500, 15)
Index(['review_id', 'review_text', 'star_rating', 'business_category',
       'platform', 'aspects', 'aspect_sentiments', 'clean_text', 'len_before',
       'len_after', 'aspects_parsed', 'sentiments_parsed', 'asp_labels',
       'sent_labels', 'model_input'],
      dtype='object')


In [22]:
ASPECTS = [
    "food","service","price","cleanliness",
    "delivery","ambiance","app_experience",
    "general","none"
]

SENTIMENTS = ["positive","negative","neutral"]

ASP2IDX = {a:i for i,a in enumerate(ASPECTS)}
SENT2IDX = {s:i for i,s in enumerate(SENTIMENTS)}

In [23]:
Y_train_aspects = np.stack(train["asp_labels"].values)
Y_train_sentiments = np.stack(train["sent_labels"].values)

Y_val_aspects = np.stack(val["asp_labels"].values)
Y_val_sentiments = np.stack(val["sent_labels"].values)

In [24]:
aspect_counts = Y_train_aspects.sum(axis=0)
max_count = aspect_counts.max()

aspect_weights = torch.tensor(
    [max_count / max(c, 1) for c in aspect_counts],
    dtype=torch.float
)

aspect_weights = torch.clamp(aspect_weights, max=5.0)

print(aspect_weights)

tensor([2.1762, 1.0000, 2.7910, 5.0000, 5.0000, 2.6138, 2.1810, 3.2607, 5.0000])


In [25]:
MODEL_NAME = "UBC-NLP/MARBERTv2"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

In [26]:
class ABSADataset(Dataset):
    def __init__(self, df):
        self.texts = df["model_input"].values
        self.aspects = np.stack(df["asp_labels"].values)
        self.sentiments = np.stack(df["sent_labels"].values)

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = tokenizer(
            str(self.texts[idx]),
            padding="max_length",
            truncation=True,
            max_length=128,
            return_tensors="pt"
        )

        return {
            "input_ids": enc["input_ids"].squeeze(),
            "attention_mask": enc["attention_mask"].squeeze(),
            "aspects": torch.tensor(self.aspects[idx], dtype=torch.float),
            "sentiments": torch.tensor(self.sentiments[idx], dtype=torch.long),
        }

In [27]:
train_ds = ABSADataset(train)
val_ds   = ABSADataset(val)

train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=16)

In [28]:
class ABSA_Model(nn.Module):
    def __init__(self):
        super().__init__()

        self.encoder = AutoModel.from_pretrained(MODEL_NAME)
        hidden = 768

        self.dropout = nn.Dropout(0.3)

        # Aspect detection (multi-label)
        self.aspect_head = nn.Linear(hidden, 9)

        # Sentiment (per aspect, 3 classes)
        self.sent_head = nn.Linear(hidden, 9 * 3)

    def forward(self, input_ids, attention_mask):

        out = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        cls = out.last_hidden_state[:, 0]
        cls = self.dropout(cls)

        aspect_logits = self.aspect_head(cls)

        sent_logits = self.sent_head(cls)
        sent_logits = sent_logits.view(-1, 9, 3)

        return aspect_logits, sent_logits

In [29]:
aspect_loss_fn = nn.BCEWithLogitsLoss(
    pos_weight=aspect_weights
)

sent_loss_fn = nn.CrossEntropyLoss(ignore_index=-1)

In [30]:
def compute_loss(asp_logits, sent_logits, y_asp, y_sent):

    loss_asp = aspect_loss_fn(asp_logits, y_asp)

    sent_logits = sent_logits.view(-1, 9, 3)

    # flatten only valid positions
    loss_sent = 0
    valid_count = 0

    for i in range(9):
        mask = (y_sent[:, i] != -1)

        if mask.sum() == 0:
            continue

        loss_sent += sent_loss_fn(
            sent_logits[mask, i, :],
            y_sent[mask, i]
        )
        valid_count += 1

    if valid_count > 0:
        loss_sent = loss_sent / valid_count
    else:
        loss_sent = 0

    return loss_asp + loss_sent

In [31]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model = ABSA_Model().to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr = 1e-5)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: UBC-NLP/MARBERTv2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [32]:
print("Max aspect weight:", aspect_weights.max())
print("Any NaN in train:", np.isnan(Y_train_aspects).any())
print("Any empty sentiment rows:", np.sum((Y_train_sentiments == -1).all(axis=1)))

Max aspect weight: tensor(5.)
Any NaN in train: False
Any empty sentiment rows: 0


In [ ]:
for epoch in range(3):

    print(f"\n🚀 ================= EPOCH {epoch+1} START =================")

    model.train()
    total_loss = 0

    for step, batch in enumerate(train_loader):

        # 🔍 CHECK 1: batch loading
        if step == 0:
            print("✅ First batch loaded successfully")
            print("Input shape:", batch["input_ids"].shape)

        input_ids = batch["input_ids"].to(device)
        mask = batch["attention_mask"].to(device)
        y_asp = batch["aspects"].to(device)
        y_sent = batch["sentiments"].to(device)

        optimizer.zero_grad()

        asp_logits, sent_logits = model(input_ids, mask)

        # 🔍 CHECK 2: forward pass sanity
        if step == 0:
            print("✅ Forward pass OK")
            print("Aspect logits:", asp_logits.shape)
            print("Sent logits:", sent_logits.shape)

        loss = compute_loss(asp_logits, sent_logits, y_asp, y_sent)

        # 🔍 CHECK 3: loss sanity
        if step % 20 == 0:
            print(f"Step {step} | Loss: {loss.item():.4f}")

        if torch.isnan(loss):
            print("❌ NaN LOSS DETECTED — stopping training")
            break

        loss.backward()

        # 🔍 CHECK 4: gradient explosion check
        total_norm = 0
        for p in model.parameters():
            if p.grad is not None:
                param_norm = p.grad.data.norm(2)
                total_norm += param_norm.item() ** 2
        total_norm = total_norm ** 0.5

        if step % 50 == 0:
            print(f"📊 Grad norm: {total_norm:.4f}")

        optimizer.step()

        total_loss += loss.item()

        # 🔍 CHECK 5: stuck detection
        if step == 0:
            first_loss = loss.item()
        if step == 100:
            if abs(loss.item() - first_loss) < 1e-4:
                print("⚠️ WARNING: model might be stuck (loss not changing)")

    print(f"🔥 Epoch {epoch+1} finished | Avg Loss: {total_loss/len(train_loader):.4f}")


🚀 ================= EPOCH 1 START =================
✅ First batch loaded successfully
Input shape: torch.Size([16, 128])
✅ Forward pass OK
Aspect logits: torch.Size([16, 9])
Sent logits: torch.Size([16, 9, 3])
Step 0 | Loss: 2.1011
📊 Grad norm: 6.6243
Step 20 | Loss: 1.7885
Step 40 | Loss: 1.6254
📊 Grad norm: 5.3837
Step 60 | Loss: 1.6236
Step 80 | Loss: 1.4275
Step 100 | Loss: 1.4370
📊 Grad norm: 4.9117
Step 120 | Loss: 1.1095
🔥 Epoch 1 finished | Avg Loss: 1.5403

🚀 ================= EPOCH 2 START =================
✅ First batch loaded successfully
Input shape: torch.Size([16, 128])
✅ Forward pass OK
Aspect logits: torch.Size([16, 9])
Sent logits: torch.Size([16, 9, 3])
Step 0 | Loss: 1.2783
📊 Grad norm: 6.8538
